In [1]:
import os
import shutil
import matplotlib.pyplot as plt
from ultralytics import YOLO
import pandas as pd
from PIL import Image

In [2]:
# Define model name
model_name = "yolo11n"

# Create Individual Results directory structure
individual_results_dir = f"Individual Results"
os.makedirs(individual_results_dir, exist_ok=True)

# Load the model
model = YOLO(f"Individual Results/{model_name}/{model_name}.pt")

# Train with modified paths
results = model.train(
    data="CS2 Dataset/YOLO_COCO/data.yaml",
    epochs=100,
    imgsz=640,
    patience=20,
    batch=16,
    workers=0,
    name=f"train",
    cache=False,
    project=f"{individual_results_dir}/{model_name}", # Direct output to Individual Results
    exist_ok=True # Overwrite existing results
)

New https://pypi.org/project/ultralytics/8.3.250 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.249  Python-3.11.9 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=CS2 Dataset/YOLO_COCO/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=Individual Results/yolo11n/yolo11n.pt, momentum=0.937, mosaic=1.0, multi_sc

In [3]:
# Delete automatically downloaded YOLO model in root directory
os.remove(f"{model_name}.pt") 

# Load best model and perform evaluation on test set
model = YOLO(f'Individual Results/{model_name}/train/weights/best.pt')

# Test on the actual test set
metrics = model.val(data='CS2 Dataset/YOLO_COCO/data.yaml', split='test', project=f"{individual_results_dir}/{model_name}", name=f"test", exist_ok=True)

# Create Group Results directory and save metrics CSV
group_results_dir = "Group Results"
os.makedirs(group_results_dir, exist_ok=True)

# Create metrics dataframe
metrics_df = pd.DataFrame({
    'Model Name': [model_name.upper()],
    'mAP@0.5': [metrics.box.map50],
    'mAP@0.5:0.95': [metrics.box.map],
    'Precision': [metrics.box.mp],
    'Recall': [metrics.box.mr],
    'Inference speed (ms)': [metrics.speed['inference']]
})

# Save to CSV
csv_path = os.path.join(group_results_dir, f"{model_name}.csv")
metrics_df.to_csv(csv_path, index=False)
print(f"Saved metrics to {csv_path}")

# Print comparison metrics
print(f"Model: {model_name.upper()}")
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")
print(f"Inference speed: {metrics.speed['inference']:.1f}ms")

Ultralytics 8.3.249  Python-3.11.9 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 100 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.30.1 ms, read: 847.284.4 MB/s, size: 3099.1 KB)
val: Scanning C:\Users\matth\Desktop\Computer Vision 2\CS2 Dataset\YOLO_COCO\labels\test.cache... 97 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 97/97 94.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.3s/it 8.9s0.3s9
                   all         97        121      0.898      0.628      0.737      0.617
Blind-Spot Mirror (Convex)         19         19      0.931      0.895      0.936      0.843
    No Entry (One Way)         30         33      0.855      0.713      0.871      0.677
No Through Road (T-Sign)         10         10      0.864        0.7      0.879      0.731
   Pedestrian Crossing         14         18      0.851      0.3

In [4]:
# Copy best model to Models directory
models_dir = "Models"
os.makedirs(models_dir, exist_ok=True)
shutil.copy(f'Individual Results/{model_name}/train/weights/best.pt', os.path.join(models_dir, f'{model_name}.pt'))

'Models\\yolo11n.pt'